# Check distributions of features for real-world data

In [7]:
import pandas as pd
from scipy.stats import shapiro

# List the 8 features here that are NOT part of your model's input features
# (Examples below based on standard targets/metadata)
columns_to_exclude = [
    'label', 
    'months_to_conversion', 
   'progression_independent_from_relapses',
   'FutureFreezing', 'FutureFalls', 
   'max_status_longi',
   'FutureMotorFluctuations',
   'FutureDyskinesia'
]

import pandas as pd
from scipy.stats import shapiro
from statsmodels.stats.multitest import multipletests
import warnings

# Suppress minor scipy warnings for clean output
warnings.filterwarnings('ignore', category=UserWarning)

def evaluate_normality_for_rebuttal(datasets, exclude_cols=None, alpha=0.05):
    """
    Evaluates normality across multiple datasets, applies FDR correction globally, 
    and generates the statistical summary for the paper rebuttal.
    """
    if exclude_cols is None:
        exclude_cols = []
        
    results = []
    
    # 1. Collect all raw p-values across all datasets
    for dataset_name, df in datasets.items():
        # Select only numerical columns and drop excluded metadata targets
        numerical_cols = [c for c in df.select_dtypes(include=['number']).columns if c not in exclude_cols]
        
        for col in numerical_cols:
            clean_data = df[col].dropna()
            
            # Skip if not enough data OR if variance is zero (constant value)
            if len(clean_data) >= 3 and clean_data.max() > clean_data.min():
                stat, p_value = shapiro(clean_data)
                results.append({
                    'Dataset': dataset_name,
                    'Feature': col,
                    'W-Statistic': stat,
                    'p-value_raw': p_value
                })
                
    results_df = pd.DataFrame(results)
    
    # 2. Apply Multiple Comparison Correction (Benjamini-Hochberg FDR)
    if not results_df.empty:
        reject, pvals_corrected, _, _ = multipletests(
            results_df['p-value_raw'], 
            alpha=alpha, 
            method='fdr_bh'
        )
        
        results_df['p-value_corrected'] = pvals_corrected
        results_df['Is_Significant_NonNormal'] = reject # True if p_corrected < alpha
        
    # 3. Calculate Final Rebuttal Stats
    total_features = len(results_df)
    non_normal_features = results_df['Is_Significant_NonNormal'].sum()
    percentage = (non_normal_features / total_features) * 100 if total_features > 0 else 0
    
    # 4. Print Summary and Draft Text
    print("--- Final FDR-Corrected Rebuttal Statistics ---")
    print(f"Total features evaluated: {total_features}")
    print(f"Features significantly deviating from normality: {non_normal_features}")
    print(f"Percentage: {percentage:.1f}%\n")
    
    print("--- Draft Text for Reviewer ---")
    print(f"\"The analysis confirmed that {percentage:.1f}% of the evaluated features ({non_normal_features} out of {total_features}) significantly deviated from normality (FDR-corrected p < {alpha}).\"")
    
    # Return the dataframe just in case you need to inspect which features passed/failed
    return results_df

# ==========================================
# EXECUTION SCRIPT
# ==========================================

# 1. Load the data
df_ms = pd.read_csv('./ms_neuro/X.csv')
df_pd = pd.read_csv('./pd_neuro/X_neuroart_v4.csv')
df_ad = pd.read_csv('./adni/adni_baseline_only_36m.csv')

datasets_dict = {
    'MS': df_ms,
    'PD': df_pd,
    'ADNI': df_ad
}

# 3. Run the evaluation
final_results_df = evaluate_normality_for_rebuttal(datasets_dict, exclude_cols=columns_to_exclude)

--- Final FDR-Corrected Rebuttal Statistics ---
Total features evaluated: 69
Features significantly deviating from normality: 68
Percentage: 98.6%

--- Draft Text for Reviewer ---
"The analysis confirmed that 98.6% of the evaluated features (68 out of 69) significantly deviated from normality (FDR-corrected p < 0.05)."
